# Inspect LLM interaction yes/no scorer

Smoke notebook for the minimal Agent4Rec/SimUSER-style LLM baseline.

The notebook uses a tiny toy canonical dataset and a fake OpenAI-compatible client. No real API call is made. The point is to inspect:

- how a `Task` table becomes `X_train, y_train, X_test, y_test`;
- which columns the LLM scorer consumes;
- how train rows become user history;
- what final chat prompts are sent for candidate groups.

In [1]:
from __future__ import annotations

import json
import re
import sys
import tempfile
from pathlib import Path
from types import SimpleNamespace

import pandas as pd


def _add_repo_root_to_path() -> None:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "pyproject.toml").exists() and (path / "runners").exists():
            for import_path in (str(path), str(path / "src")):
                if import_path not in sys.path:
                    sys.path.insert(0, import_path)
            return
    raise RuntimeError("Could not find beyond-click-sim repo root")


_add_repo_root_to_path()

from runners.in_distribution.interaction_prediction.task_builders import repo_root

REPO_ROOT = repo_root()

print("repo:", REPO_ROOT)


repo: /Users/a.agafonov/Research/agent_simulator/repos/beyond-click-sim


In [2]:
from beyond_click_sim.data.canonical import CanonicalDataset
from beyond_click_sim.scorers import LLMInteractionYesNoScorer
from beyond_click_sim.tasks import (
    AlignmentInteractionTaskBuilder,
    MinUserInteractionsFilter,
    NonInteractionCandidateSampler,
    RandomFractionSplitter,
    split_xy,
)

## 1. Build a tiny canonical dataset

This cell creates three canonical tables: `users`, `items`, and `interactions`. It mirrors the real data shape, but stays tiny enough for prompt inspection.

In [3]:
def write_toy_canonical_dataset(root: Path) -> CanonicalDataset:
    root.mkdir(parents=True, exist_ok=True)

    users = pd.DataFrame(
        {
            "user_id": ["u_animation", "u_strategy", "u_sparse"],
            "age": [24, 33, 19],
            "segment": ["family_movies", "strategy_games", "too_few_interactions"],
        }
    )
    items = pd.DataFrame(
        {
            "item_id": [f"i{idx}" for idx in range(1, 13)],
            "title": [
                "Toy Story",
                "The Lion King",
                "Finding Nemo",
                "Shrek",
                "Civilization VI",
                "XCOM 2",
                "Into the Breach",
                "Age of Empires II",
                "The Godfather",
                "Portal 2",
                "Heat",
                "Stardew Valley",
            ],
            "genre": [
                "animation adventure",
                "animation musical",
                "animation family",
                "animation comedy",
                "turn-based strategy",
                "tactical strategy",
                "tactical puzzle",
                "real-time strategy",
                "crime drama",
                "puzzle comedy",
                "crime thriller",
                "farming sim",
            ],
        }
    )
    interactions = pd.DataFrame(
        {
            "interaction_id": [f"r{idx}" for idx in range(1, 12)],
            "user_id": [
                "u_animation",
                "u_animation",
                "u_animation",
                "u_animation",
                "u_strategy",
                "u_strategy",
                "u_strategy",
                "u_strategy",
                "u_strategy",
                "u_sparse",
                "u_sparse",
            ],
            "item_id": [
                "i1",
                "i2",
                "i3",
                "i4",
                "i5",
                "i6",
                "i7",
                "i8",
                "i10",
                "i1",
                "i12",
            ],
            "rating": [5, 4, 5, 3, 5, 4, 4, 3, 2, 4, 5],
            "target_interact": [1] * 11,
        }
    )

    users_path = root / "users.parquet"
    items_path = root / "items.parquet"
    interactions_path = root / "interactions.parquet"
    manifest_path = root / "manifest.json"

    users.to_parquet(users_path, index=False)
    items.to_parquet(items_path, index=False)
    interactions.to_parquet(interactions_path, index=False)
    manifest_path.write_text(json.dumps({"dataset": "toy-llm"}), encoding="utf-8")

    return CanonicalDataset(
        name="toy-llm",
        version="v1",
        root=root,
        users_path=users_path,
        items_path=items_path,
        interactions_path=interactions_path,
        manifest_path=manifest_path,
    )


tmp_dir = tempfile.TemporaryDirectory()
dataset = write_toy_canonical_dataset(Path(tmp_dir.name) / "toy-canonical")

display(dataset.load_users())
display(dataset.load_items())
display(dataset.load_interactions())

,user_id,age,segment
0,u_animation,24,family_movies
1,u_strategy,33,strategy_games
2,u_sparse,19,too_few_interactions


,item_id,title,genre
0,i1,Toy Story,animation adventure
1,i2,The Lion King,animation musical
2,i3,Finding Nemo,animation family
3,i4,Shrek,animation comedy
4,i5,Civilization VI,turn-based strategy
5,i6,XCOM 2,tactical strategy
6,i7,Into the Breach,tactical puzzle
7,i8,Age of Empires II,real-time strategy
8,i9,The Godfather,crime drama
9,i10,Portal 2,puzzle comedy


,interaction_id,user_id,item_id,target_interact
0,r1,u_animation,i1,1
1,r2,u_animation,i2,1
2,r3,u_animation,i3,1
3,r4,u_animation,i4,1
4,r5,u_strategy,i5,1
5,r6,u_strategy,i6,1
6,r7,u_strategy,i7,1
7,r8,u_strategy,i8,1
8,r9,u_strategy,i10,1
9,r10,u_sparse,i1,1


## 2. Materialize the interaction-alignment task

The task builder performs the usual pipeline: filter users, split observed interactions, then add sampled non-interactions to held-out positives.

For LLM yes/no scoring, `train` is the user history table. `test` is a candidate-set table grouped by `candidate_group`.

In [4]:
builder = AlignmentInteractionTaskBuilder(
    name="toy-llm-interaction-alignment-1to2",
    dataset_filter=MinUserInteractionsFilter(min_interactions=4),
    splitter=RandomFractionSplitter(
        train_fraction=0.75,
        val_fraction=0.0,
        test_fraction=0.25,
        seed=0,
        group_column="user_id",
        stratify_column=None,
    ),
    sampler=NonInteractionCandidateSampler(negative_ratio=2, seed=0),
    history_context_columns=("rating",),
)

task = builder.build(dataset)

print("task:", task.name)
print("schema:", task.schema)
display(pd.DataFrame([task.manifest["rows"]]))
display(task.train)
display(task.test)

task: toy-llm-interaction-alignment-1to2
schema: TaskSchema(target_column='target', feature_columns=('user_age', 'user_segment', 'item_title', 'item_genre'), id_columns=('user_id', 'item_id'), candidate_group_column='candidate_group', sampled_column='sampled')


,train,val,test
0,6,0,9


,user_id,item_id,user_age,user_segment,item_title,item_genre,target,sampled,candidate_group
0,u_animation,i4,24,family_movies,Shrek,animation comedy,1,False,<NA>
1,u_animation,i2,24,family_movies,The Lion King,animation musical,1,False,<NA>
2,u_animation,i1,24,family_movies,Toy Story,animation adventure,1,False,<NA>
3,u_strategy,i10,33,strategy_games,Portal 2,puzzle comedy,1,False,<NA>
4,u_strategy,i5,33,strategy_games,Civilization VI,turn-based strategy,1,False,<NA>
5,u_strategy,i8,33,strategy_games,Age of Empires II,real-time strategy,1,False,<NA>


,user_id,item_id,user_age,user_segment,item_title,item_genre,target,sampled,candidate_group
0,u_animation,i3,24,family_movies,Finding Nemo,animation family,1,False,candidate:u_animation:r3
1,u_animation,i6,24,family_movies,XCOM 2,tactical strategy,0,True,candidate:u_animation:r3
2,u_animation,i12,24,family_movies,Stardew Valley,farming sim,0,True,candidate:u_animation:r3
3,u_strategy,i7,33,strategy_games,Into the Breach,tactical puzzle,1,False,candidate:u_strategy:r7
4,u_strategy,i2,33,strategy_games,The Lion King,animation musical,0,True,candidate:u_strategy:r7
5,u_strategy,i1,33,strategy_games,Toy Story,animation adventure,0,True,candidate:u_strategy:r7
6,u_strategy,i6,33,strategy_games,XCOM 2,tactical strategy,1,False,candidate:u_strategy:r6
7,u_strategy,i12,33,strategy_games,Stardew Valley,farming sim,0,True,candidate:u_strategy:r6
8,u_strategy,i2,33,strategy_games,The Lion King,animation musical,0,True,candidate:u_strategy:r6


## 3. Split task tables into X/y

`split_xy` removes only the target column. The scorer decides which input columns it needs. For this LLM baseline those are:

- `user_id`;
- `candidate_group` during scoring;
- `history_description_columns`, used only for fitted user history;
- `candidate_description_columns`, used only for scored candidate rows.

Keeping these lists separate avoids leakage when history can show known feedback, such as ratings, but candidates must not expose target values. In this notebook, `rating` is copied into train history rows and set to missing in test candidate rows.

In [5]:
X_train, y_train = split_xy(task.train, target_column=task.schema.target_column)
X_test, y_test = split_xy(task.test, target_column=task.schema.target_column)

history_description_columns = ("item_title", "item_genre", "rating")
candidate_description_columns = ("item_title", "item_genre")

print("X_train columns:", list(X_train.columns))
print("X_test columns:", list(X_test.columns))
print("y_train name:", y_train.name)
print("y_test name:", y_test.name)

display(X_train[["user_id", "item_id", *history_description_columns, "sampled", "candidate_group"]])
display(X_test[["user_id", "item_id", *candidate_description_columns, "rating", "sampled", "candidate_group"]])

X_train columns: ['user_id', 'item_id', 'user_age', 'user_segment', 'item_title', 'item_genre', 'sampled', 'candidate_group']
X_test columns: ['user_id', 'item_id', 'user_age', 'user_segment', 'item_title', 'item_genre', 'sampled', 'candidate_group']
y_train name: target
y_test name: target


,user_id,item_id,item_title,item_genre,sampled,candidate_group
0,u_animation,i4,Shrek,animation comedy,False,<NA>
1,u_animation,i2,The Lion King,animation musical,False,<NA>
2,u_animation,i1,Toy Story,animation adventure,False,<NA>
3,u_strategy,i10,Portal 2,puzzle comedy,False,<NA>
4,u_strategy,i5,Civilization VI,turn-based strategy,False,<NA>
5,u_strategy,i8,Age of Empires II,real-time strategy,False,<NA>


,user_id,item_id,item_title,item_genre,sampled,candidate_group
0,u_animation,i3,Finding Nemo,animation family,False,candidate:u_animation:r3
1,u_animation,i6,XCOM 2,tactical strategy,True,candidate:u_animation:r3
2,u_animation,i12,Stardew Valley,farming sim,True,candidate:u_animation:r3
3,u_strategy,i7,Into the Breach,tactical puzzle,False,candidate:u_strategy:r7
4,u_strategy,i2,The Lion King,animation musical,True,candidate:u_strategy:r7
5,u_strategy,i1,Toy Story,animation adventure,True,candidate:u_strategy:r7
6,u_strategy,i6,XCOM 2,tactical strategy,False,candidate:u_strategy:r6
7,u_strategy,i12,Stardew Valley,farming sim,True,candidate:u_strategy:r6
8,u_strategy,i2,The Lion King,animation musical,True,candidate:u_strategy:r6


## 4. Fake OpenAI-compatible client

The scorer calls `client.chat.completions.create(...)`. This fake client records the final messages and returns deterministic yes/no lines, so we can inspect prompts without paying for an API call.

In [6]:
class RecordingChatCompletions:
    def __init__(self) -> None:
        self.calls: list[dict[str, object]] = []

    def create(self, **kwargs: object):
        self.calls.append(kwargs)
        user_prompt = kwargs["messages"][1]["content"]
        labels = re.findall(r"^C\d+(?=\.)", user_prompt, flags=re.MULTILINE)
        response_lines = [
            f"{label}: {'yes' if idx == 0 else 'no'}"
            for idx, label in enumerate(labels)
        ]
        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content="\n".join(response_lines)))]
        )


class RecordingClient:
    def __init__(self) -> None:
        completions = RecordingChatCompletions()
        self.chat = SimpleNamespace(completions=completions)
        self.completions = completions


def print_chat_call(call: dict[str, object]) -> None:
    messages = call["messages"]
    print("MODEL:", call["model"])
    print("TEMPERATURE:", call["temperature"])
    print("MAX TOKENS:", call["max_tokens"])
    print("\n--- SYSTEM ---")
    print(messages[0]["content"])
    print("\n--- USER ---")
    print(messages[1]["content"])

## 5. Fit and score with capped history

Here `max_history_items=2`, so each user's prompt history contains at most the latest two train rows for that user.

In [7]:
client = RecordingClient()
scorer = LLMInteractionYesNoScorer(
    client=client,
    model="fake-llm",
    history_description_columns=history_description_columns,
    candidate_description_columns=candidate_description_columns,
    max_history_items=2,
    temperature=0.0,
    max_tokens=128,
)

scorer.fit(X_train, y_train)
scores = scorer.score(X_test)

display(scorer.history_by_user_)
display(pd.concat([X_test[["user_id", "item_id", *candidate_description_columns, "candidate_group"]], y_test, scores], axis=1))
print("chat calls:", len(client.completions.calls))

[{'role': 'system', 'content': "You are simulating a user's response in a recommender system.\nGiven the user's interaction history and candidate items, decide whether the user would interact with each candidate.\nUse only the provided history and candidate information.\nReturn only one line per candidate in the required format."}, {'role': 'user', 'content': 'User interaction history:\nH1. item_title: Shrek; item_genre: animation comedy\nH2. item_title: The Lion King; item_genre: animation musical\n\nCandidate items:\nC1. item_title: Finding Nemo; item_genre: animation family\nC2. item_title: XCOM 2; item_genre: tactical strategy\nC3. item_title: Stardew Valley; item_genre: farming sim\n\nFor each candidate, answer whether the user would interact with it.\n\nRequired output format:\nReturn exactly one line for each candidate label.\nThe text before each colon must match these labels:\nC1:\nC2:\nC3:\nFill each line after the colon with either yes or no.'}]
namespace(choices=[namespace(

{'u_animation': ['H1. item_title: Shrek; item_genre: animation comedy',
  'H2. item_title: The Lion King; item_genre: animation musical'],
 'u_strategy': ['H1. item_title: Portal 2; item_genre: puzzle comedy',
  'H2. item_title: Civilization VI; item_genre: turn-based strategy']}

,user_id,item_id,item_title,item_genre,candidate_group,target,score
0,u_animation,i3,Finding Nemo,animation family,candidate:u_animation:r3,1,1.0
1,u_animation,i6,XCOM 2,tactical strategy,candidate:u_animation:r3,0,0.0
2,u_animation,i12,Stardew Valley,farming sim,candidate:u_animation:r3,0,0.0
3,u_strategy,i7,Into the Breach,tactical puzzle,candidate:u_strategy:r7,1,1.0
4,u_strategy,i2,The Lion King,animation musical,candidate:u_strategy:r7,0,0.0
5,u_strategy,i1,Toy Story,animation adventure,candidate:u_strategy:r7,0,0.0
6,u_strategy,i6,XCOM 2,tactical strategy,candidate:u_strategy:r6,1,1.0
7,u_strategy,i12,Stardew Valley,farming sim,candidate:u_strategy:r6,0,0.0
8,u_strategy,i2,The Lion King,animation musical,candidate:u_strategy:r6,0,0.0


chat calls: 3


### Prompt variant A: capped history prompt

This is the exact first chat request produced by `score(X_test)`.

In [8]:
print_chat_call(client.completions.calls[0])

MODEL: fake-llm
TEMPERATURE: 0.0
MAX TOKENS: 128

--- SYSTEM ---
You are simulating a user's response in a recommender system.
Given the user's interaction history and candidate items, decide whether the user would interact with each candidate.
Use only the provided history and candidate information.
Return only one line per candidate in the required format.

--- USER ---
User interaction history:
H1. item_title: Shrek; item_genre: animation comedy
H2. item_title: The Lion King; item_genre: animation musical

Candidate items:
C1. item_title: Finding Nemo; item_genre: animation family
C2. item_title: XCOM 2; item_genre: tactical strategy
C3. item_title: Stardew Valley; item_genre: farming sim

For each candidate, answer whether the user would interact with it.

Required output format:
Return exactly one line for each candidate label.
The text before each colon must match these labels:
C1:
C2:
C3:
Fill each line after the colon with either yes or no.


## 6. Prompt variant B: full history with `max_history_items=None`

Same candidate group, but the scorer keeps all train rows for the user.

In [9]:
first_group = X_test[task.schema.candidate_group_column].iloc[0]
X_one_group = X_test[X_test[task.schema.candidate_group_column].eq(first_group)]

full_history_client = RecordingClient()
full_history_scorer = LLMInteractionYesNoScorer(
    client=full_history_client,
    model="fake-llm",
    history_description_columns=history_description_columns,
    candidate_description_columns=candidate_description_columns,
    max_history_items=None,
)
full_history_scorer.fit(X_train, y_train)
full_history_scorer.score(X_one_group)

print_chat_call(full_history_client.completions.calls[0])

[{'role': 'system', 'content': "You are simulating a user's response in a recommender system.\nGiven the user's interaction history and candidate items, decide whether the user would interact with each candidate.\nUse only the provided history and candidate information.\nReturn only one line per candidate in the required format."}, {'role': 'user', 'content': 'User interaction history:\nH1. item_title: Shrek; item_genre: animation comedy\nH2. item_title: The Lion King; item_genre: animation musical\nH3. item_title: Toy Story; item_genre: animation adventure\n\nCandidate items:\nC1. item_title: Finding Nemo; item_genre: animation family\nC2. item_title: XCOM 2; item_genre: tactical strategy\nC3. item_title: Stardew Valley; item_genre: farming sim\n\nFor each candidate, answer whether the user would interact with it.\n\nRequired output format:\nReturn exactly one line for each candidate label.\nThe text before each colon must match these labels:\nC1:\nC2:\nC3:\nFill each line after the c

## 7. Prompt variant C: cold user / no fitted history

If `score(X)` contains a user absent from `fit(X_train, y_train)`, the prompt explicitly says that no interaction history is available.

In [10]:
cold_X = pd.DataFrame(
    {
        "user_id": ["u_cold", "u_cold"],
        "item_id": ["i9", "i12"],
        "user_age": [pd.NA, pd.NA],
        "user_segment": [pd.NA, pd.NA],
        "item_title": ["The Godfather", "Stardew Valley"],
        "item_genre": ["crime drama", "farming sim"],
        "sampled": [False, True],
        "candidate_group": ["manual:cold-user", "manual:cold-user"],
    },
    index=["cold_positive", "cold_negative"],
)

cold_client = RecordingClient()
cold_scorer = LLMInteractionYesNoScorer(
    client=cold_client,
    model="fake-llm",
    history_description_columns=history_description_columns,
    candidate_description_columns=candidate_description_columns,
    max_history_items=2,
)
cold_scorer.fit(X_train, y_train)
cold_scores = cold_scorer.score(cold_X)

display(pd.concat([cold_X, cold_scores], axis=1))
print_chat_call(cold_client.completions.calls[0])

[{'role': 'system', 'content': "You are simulating a user's response in a recommender system.\nGiven the user's interaction history and candidate items, decide whether the user would interact with each candidate.\nUse only the provided history and candidate information.\nReturn only one line per candidate in the required format."}, {'role': 'user', 'content': 'User interaction history:\n- No interaction history available.\n\nCandidate items:\nC1. item_title: The Godfather; item_genre: crime drama\nC2. item_title: Stardew Valley; item_genre: farming sim\n\nFor each candidate, answer whether the user would interact with it.\n\nRequired output format:\nReturn exactly one line for each candidate label.\nThe text before each colon must match these labels:\nC1:\nC2:\nFill each line after the colon with either yes or no.'}]
namespace(choices=[namespace(message=namespace(content='C1: yes\nC2: no'))])


,user_id,item_id,user_age,user_segment,item_title,item_genre,sampled,candidate_group,score
cold_positive,u_cold,i9,<NA>,<NA>,The Godfather,crime drama,False,manual:cold-user,1.0
cold_negative,u_cold,i12,<NA>,<NA>,Stardew Valley,farming sim,True,manual:cold-user,0.0


MODEL: fake-llm
TEMPERATURE: 0.0
MAX TOKENS: 512

--- SYSTEM ---
You are simulating a user's response in a recommender system.
Given the user's interaction history and candidate items, decide whether the user would interact with each candidate.
Use only the provided history and candidate information.
Return only one line per candidate in the required format.

--- USER ---
User interaction history:
- No interaction history available.

Candidate items:
C1. item_title: The Godfather; item_genre: crime drama
C2. item_title: Stardew Valley; item_genre: farming sim

For each candidate, answer whether the user would interact with it.

Required output format:
Return exactly one line for each candidate label.
The text before each colon must match these labels:
C1:
C2:
Fill each line after the colon with either yes or no.


## 8. Local Ollama smoke: `llama3.1:8b`

This cell passes the local Ollama OpenAI-compatible client directly into the scorer. It scores only the first candidate group to keep the smoke test cheap.

In [11]:
from beyond_click_sim.llm_clients import ollama_client

OLLAMA_MODEL = "llama3.1:8b"
ollama_group = X_test[task.schema.candidate_group_column].iloc[0]
X_ollama = X_test[X_test[task.schema.candidate_group_column].eq(ollama_group)]

ollama_scorer = LLMInteractionYesNoScorer(
    client=ollama_client(),
    model=OLLAMA_MODEL,
    history_description_columns=history_description_columns,
    candidate_description_columns=candidate_description_columns,
    max_history_items=2,
    temperature=0.0,
    max_tokens=64,
)

try:
    ollama_scorer.fit(X_train, y_train)
    ollama_scores = ollama_scorer.score(X_ollama)
    display(pd.concat([X_ollama[["user_id", "item_id", *candidate_description_columns, "candidate_group"]], ollama_scores], axis=1))
except Exception as error:
    print("Ollama smoke failed:", repr(error))

[{'role': 'system', 'content': "You are simulating a user's response in a recommender system.\nGiven the user's interaction history and candidate items, decide whether the user would interact with each candidate.\nUse only the provided history and candidate information.\nReturn only one line per candidate in the required format."}, {'role': 'user', 'content': 'User interaction history:\nH1. item_title: Shrek; item_genre: animation comedy\nH2. item_title: The Lion King; item_genre: animation musical\n\nCandidate items:\nC1. item_title: Finding Nemo; item_genre: animation family\nC2. item_title: XCOM 2; item_genre: tactical strategy\nC3. item_title: Stardew Valley; item_genre: farming sim\n\nFor each candidate, answer whether the user would interact with it.\n\nRequired output format:\nReturn exactly one line for each candidate label.\nThe text before each colon must match these labels:\nC1:\nC2:\nC3:\nFill each line after the colon with either yes or no.'}]
ChatCompletion(id='chatcmpl-5

,user_id,item_id,item_title,item_genre,candidate_group,score
0,u_animation,i3,Finding Nemo,animation family,candidate:u_animation:r3,0.0
1,u_animation,i6,XCOM 2,tactical strategy,candidate:u_animation:r3,0.0
2,u_animation,i12,Stardew Valley,farming sim,candidate:u_animation:r3,0.0


## 9. Real client sketch

Use this only when `.env` contains `OPENAI_API_KEY`. Model names and custom endpoints stay explicit in code/notebooks; secrets stay in `.env`.

```python
from beyond_click_sim.llm_clients import openai_client, vllm_client, ollama_client

client = openai_client()
# client = vllm_client(base_url="http://127.0.0.1:8000/v1")
# client = ollama_client()

real_scorer = LLMInteractionYesNoScorer(
    client=client,
    model="gpt-4.1-mini",
    history_description_columns=("item_title", "item_genre", "rating"),
    candidate_description_columns=("item_title", "item_genre"),
    max_history_items=30,
)

real_scorer.fit(X_train, y_train)
real_scores = real_scorer.score(X_test.head(6))
```